##¿CUÁL ES EL VERDADERO COSTO DE LOS DERRRAMES DE PETRÓLEO EN MÉXICO?

In [ ]:
from IPython.display import HTML, display

# 1. Enlaces
link_del_mapa = "https://www.google.com/maps/d/u/0/viewer?mid=1Ku4wk-WcKmPcrna08enfELqsr5pp9LA&ll=22.865042981462132%2C-102.49244999494258&z=5"
url_imagen_captura = "https://raw.githubusercontent.com/nodracPM/HackODS_LosOutliers/main/imagenes/corredorArrecifal.png"

# 2. Código con redacción mejorada
tarjeta_html = f"""
<div style="background-color: #ffffff; border: 1px solid #d3d3d3; border-radius: 12px; padding: 25px; max-width: 650px; margin: 0 auto; box-shadow: 0 4px 8px rgba(0,0,0,0.1); font-family: sans-serif;">

    <div style="text-align: center;">
        <a href="{link_del_mapa}" target="_blank">
            <img src="{url_imagen_captura}" style="width: 100%; max-width: 500px; border-radius: 8px; border: 1px solid #eeeeee; cursor: pointer;">
        </a>
    </div>

    <div style="text-align: center; margin-top: 15px;">
        <a href="{link_del_mapa}" target="_blank" style="text-decoration: none; font-size: 14px; color: #0055ff; font-weight: bold;">
            TOCA AQUÍ PARA EXPLORAR MAPA COMUNITARIO (RED CORREDOR ARRECIFAL)
        </a>
    </div>

    <hr style="border: 0; height: 1px; background-color: #e0e0e0; margin: 20px 0;">

    <h3 style="color: #1a1a1a; margin-top: 0; text-align: center; font-size: 19px;">
        Democratización de Datos ante Los Derrames de Petróleo
    </h3>

    <p style="font-size: 14px; color: #333333; line-height: 1.6; text-align: justify; margin-bottom: 12px;">
        Este mapa, construido por la <b>Red Corredor Arrecifal</b>, representa el grito de auxilio de la ciudadanía y la vida marina ante el derrame de petróleo de marzo de 2026 en el Golfo de México. Nuestra misión es <b>dar voz a a la emergencia de los derrames de petróleo a través de los datos</b>; nos sumamos a este esfuerzo con el objetivo de democratizar la información técnica y así responder una pregunta fundamental: ¿cuál ha sido el verdadero costo de los derrames de petróleo en México a través del tiempo?
    </p>

    <p style="font-size: 14px; color: #333333; line-height: 1.6; text-align: justify; margin-bottom: 0;">
        Tomamos este suceso como el detonante para observar datos históricos multidimensionales.
    </p>
</div>
"""

# 3. Mostrar la tarjeta
display(HTML(tarjeta_html))

Comenzamos importando las bases de datos a trabajar

In [ ]:
import pandas as pd
df_semarnat = pd.read_csv('../data/d2_industri02_08.csv', encoding='latin-1')

#  Rellenamos los años hacia abajo para deshacer las celdas combinadas
df_semarnat['Año'] = df_semarnat['Año'].ffill()

#  Forzamos a que el año sea un número entero
df_semarnat['Año'] = df_semarnat['Año'].astype(int)

# Limpiamos espacios en blanco extraños en los textos
df_semarnat['Organismo Subsidiario'] = df_semarnat['Organismo Subsidiario'].astype(str).str.strip()

# Filtramos las filas que dicen "Total"
df_semarnat_totales = df_semarnat[df_semarnat['Organismo Subsidiario'].str.contains('total', case=False, na=False)].copy()

# Borramos la columna de organismo porque ya todas son el total
df_semarnat_totales = df_semarnat_totales.drop(columns=['Organismo Subsidiario'])

# Mostramos la tabla limpia de derrames
print("BASE DE SEMARNAT LIMPIA ")
display(df_semarnat_totales)

BASE DE SEMARNAT LIMPIA 


,Año,Número de derrames,Cantidad reportada (toneladas)
2,1999,856,"3,447.00"
5,2000,"1,518","6,252.00"
8,2001,523,"5,762.00"
13,2002,378,"19,807.00"
18,2003,431,"4,824.00"
23,2004,517,"5,478.00"
28,2005,359,"2,321.00"
33,2006,277,"3,432.92"
36,2007,206,"1,439.00"
39,2008,171,"2,327.00"


In [ ]:
import pandas as pd
import numpy as np
# Cargamos el archivo del INEGI
df_CEEM = pd.read_csv('../data/SCEEM_20.csv', encoding='latin-1')

# Desdoblamos la tabla de formato ancho a largo (Melt)
df_limpio = df_CEEM.melt(id_vars=['Concepto'], var_name='Columna_Fea', value_name='Valor')

# Partimos el título espantoso usando el "|" como separador (Split)
df_limpio[['Anio', 'Texto_Inutil', 'Sector']] = df_limpio['Columna_Fea'].str.split('|', expand=True)

# Borramos la columna de en medio que no aporta nada y la original fea
df_limpio = df_limpio.drop(columns=['Texto_Inutil', 'Columna_Fea'])

# Limpiamos los espacios invisibles que deja el INEGI
df_limpio['Anio'] = df_limpio['Anio'].str.strip()
df_limpio['Sector'] = df_limpio['Sector'].str.strip()
df_limpio['Concepto'] = df_limpio['Concepto'].str.strip()


# Filtramos SOLO el sector minería (Pemex)
df_pemex = df_limpio[df_limpio['Sector'].str.contains('21 Minería', na=False)].copy()

conceptos_clave = [
    'Producto interno bruto',
    'Gasto en Protección Ambiental total',
    'Costos Totales por Agotamiento y Degradación (CTADA)',
    'Agotamiento de recursos naturales',
    'Hidrocarburos', # El impacto directo de PEMEX
    'Degradación del medio ambiente', # Derrames, contaminación de agua/suelo
    'Producto interno neto ajustado por agotamiento'
]

df_final_inegi = df_pemex[df_pemex['Concepto'].isin(conceptos_clave)].copy()




# Quitamos comas si existen y forzamos a que sean números reales
df_final_inegi['Valor'] = df_final_inegi['Valor'].astype(str).str.replace(',', '')
df_final_inegi['Valor'] = pd.to_numeric(df_final_inegi['Valor'], errors='coerce')


# Transformamos conceptos en columnas por año
df_lista_para_graficar = df_final_inegi.pivot_table(
    index='Anio',
    columns='Concepto',
    values='Valor',
    aggfunc='sum'
).reset_index()

# Vemos la tabla final lista para el dashboard
display(df_lista_para_graficar.head(10))

Concepto,Anio,Agotamiento de recursos naturales,Degradación del medio ambiente,Hidrocarburos,Producto interno bruto,Producto interno neto ajustado por agotamiento
0,2003,41579.40,82.563,41579.40,406555.214,325913.761
1,2004,31249.52,83.579,31249.52,568954.863,493456.365
2,2005,44596.76,86.949,44596.76,701964.432,610424.552
3,2006,55989.72,90.375,55989.72,803347.034,694596.271
4,2007,63967.68,92.931,63967.68,911657.126,787426.066
5,2008,74441.43,93.676,74441.43,1052693.161,903248.446
6,2009,53895.44,95.566,53895.44,782592.250,631415.582
7,2010,66021.57,109.064,66021.57,956298.438,773600.871
8,2011,90171.20,108.846,90171.20,1276203.965,1057966.263
9,2012,89703.90,112.220,89703.90,1288603.295,1057223.572


## SECCIÓN COMPARATIVA DE COSTOS DE REMEDIACIÓN Y DÉFICIT ECOLÓGICO
---

### Métrica 1: Escenario de Restauración Integral (Modelo BOSCEM - US EPA)
**Fuente:** *[Modeling Oil Spill Response and Damage Costs, Etkin, 2004](https://archive.epa.gov/emergencies/docs/oil/fss/fss04/web/pdf/etkin2_04.pdf)*

Para cuantificar el impacto económico real de los derrames de **Crudo Maya**, utilizamos el modelo BOSCEM de la EPA. Utilizamos la clasificación de *Heavy Oils* porque el modelo indica explícitamente que los crudos pesados pertenecen a esta categoría. Utilizar una métrica de crudo genérico ignoraría la persistencia ambiental del fluido y subestimaría gravemente la remoción mecánica.

La fórmula integral del daño por galón se define como:
$$C_{total} = C_{respuesta} + D_{socioeconómico} + D_{ambiental}$$

* **1. Costo de Respuesta ($C_{resp}$):** $C_{resp} = \$77.00 \, \text{USD} \times 1.0 = \mathbf{\$77.00 \, \text{USD/gal}}$
* **2. Daños Socioeconómicos ($D_{soc}$):** $D_{soc} = \$175.00 \, \text{USD} \times 1.0 = \mathbf{\$175.00 \, \text{USD/gal}}$
* **3. Daños Ambientales ($D_{env}$):** $D_{env} = \$35.00 \, \text{USD} \times [0.5 \times (0.9 + 1.2)] = \mathbf{\$36.75 \, \text{USD/gal}}$

**Costo Total Integral:**
$$C_{total} = \$77.00 + \$175.00 + \$36.75 = \mathbf{\$288.75 \, \text{USD/gal}}$$

Considerando que 1 tonelada métrica de crudo pesado Maya equivale a **294 galones**, el costo real de remediación es:
$$\text{Costo por Tonelada} = \$288.75 \times 294 \approx \mathbf{\$84,892.50 \, \text{USD/Ton}}$$

---

### Métrica 2: Escenario Conservador (Costo Base Histórico)
**Fuente:** *[A Taxonomy of Oil Spill Costs, Cohen, 2010](https://media.rff.org/documents/RFF-BCK-Cohen-DHCosts.pdf)*

Basado en el análisis de reclamaciones globales reales, el promedio histórico que las empresas terminan pagando en la corte (tras deducciones y negociaciones) es de **$16 USD por galón**. Utilizando la conversión de crudo estándar (310 galones por tonelada):
$$\text{Costo Histórico} = \$16 \, \text{USD/gal} \times 310 \approx \mathbf{\$4,960 \, \text{USD/Ton}}$$

> **Nota Crítica:** En las visualizaciones de este Dashboard, para mantener un criterio **ultraconservador**, utilizaremos un piso mínimo redondeado de **$5,000 USD/Ton**. Como se observa en la Métrica 1, el costo real validado por la EPA podría ser hasta **17 veces mayor** al valor presentado en nuestras gráficas.

---

### Métrica 3: Déficit de Inversión y Daño Real (INEGI)
**Fuente:** *Sistema de Cuentas Económicas y Ecológicas de México (INEGI)*

Para contrastar el costo de los derrames con la capacidad de respuesta del Estado Mexicano, utilizamos los indicadores macroeconómicos del INEGI:
* **GPA (Gastos en Protección Ambiental):** Presupuesto real que el gobierno destina a la protección y remediación ecológica.
* **CTADA (Costos Totales por Agotamiento y Degradación):** El impacto económico total reconocido por la pérdida de recursos y calidad ambiental.

El ratio **GPA/CTADA** se utiliza en nuestro análisis como el *Índice de Insuficiencia*, demostrando estadísticamente qué porcentaje del daño ecológico anual está siendo cubierto por la inversión federal.

---

### Métrica 4: Tope Legal de Sanción (LFRA)
**Fuente:** *Ley Federal de Responsabilidad Ambiental (Art. 19)*

Representa el castigo económico máximo aplicable a Pemex. La ley establece una multa máxima de **600,000 UMAs** para personas morales. Al valor actual, esto representa un tope duro de aproximadamente **65.1 Millones de MXN**.
Además, la legislación dicta que de esta misma bolsa se deben deducir los gastos legales del demandante (erogaciones), reduciendo aún más el capital real disponible para la restauración del ecosistema.

In [ ]:
import pandas as pd
import numpy as np


# 1. PREPARAMOS  Database de SEMARNAT (Añadiendo ITOPF y EPA)
df_dashboard = df_semarnat_totales.copy()
df_dashboard['Cantidad reportada (toneladas)'] = df_dashboard['Cantidad reportada (toneladas)'].astype(str).str.replace(',', '', regex=True)
df_dashboard['Cantidad reportada (toneladas)'] = pd.to_numeric(df_dashboard['Cantidad reportada (toneladas)'], errors='coerce')
df_dashboard = df_dashboard.rename(columns={'Año': 'año'})
df_dashboard['año'] = df_dashboard['año'].astype(int)

tipo_de_cambio = 20
# MÉTRICA 2: Conservador (ITOPF)
df_dashboard['Costo Limpieza ITOPF (Millones MXN)'] = (df_dashboard['Cantidad reportada (toneladas)'] * 5000 * tipo_de_cambio) / 1000000

# MÉTRICA 1: Integral (EPA) - ¡El Monstruo!
df_dashboard['Costo Restauración EPA (Millones MXN)'] = (df_dashboard['Cantidad reportada (toneladas)'] * 84892.50 * tipo_de_cambio) / 1000000


# 2. PREPARAMOS Database de INEGI

df_raw_inegi = pd.read_csv('../data/SCEEM_20.csv', encoding='latin-1')
df_melt = df_raw_inegi.melt(id_vars=['Concepto'], var_name='Columna_Fea', value_name='Valor')
df_melt[['año', 'basura', 'Sector']] = df_melt['Columna_Fea'].str.split('|', expand=True)

df_melt['año'] = pd.to_numeric(df_melt['año'].str.strip(), errors='coerce')
df_melt['Valor'] = pd.to_numeric(df_melt['Valor'].astype(str).str.replace(',', '', regex=True), errors='coerce')

# Filtramos GPA
df_gpa = df_melt[df_melt['Concepto'].str.contains('protección ambiental', case=False, na=False)].copy()
df_inegi_limpio = df_gpa.groupby('año')['Valor'].sum().reset_index()
df_inegi_limpio = df_inegi_limpio.rename(columns={'Valor': 'Gasto Protección Ambiental (Millones MXN)'})

# MÉTRICA 4: Tope Legal (LFRA)
df_inegi_limpio['Multa Máxima LFRA (Millones MXN)'] = 65.1


# 3. Realizamos merge
df_triangulo = pd.merge(df_dashboard, df_inegi_limpio, on='año', how='inner')

columnas_finales = [
    'año',
    'Cantidad reportada (toneladas)',
    'Gasto Protección Ambiental (Millones MXN)',
    'Multa Máxima LFRA (Millones MXN)',
    'Costo Limpieza ITOPF (Millones MXN)',
    'Costo Restauración EPA (Millones MXN)'
]

display(df_triangulo[columnas_finales].head())

: 

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# 1. Gasto del Gobierno (Verde)
fig.add_trace(go.Bar(
    x=df_triangulo['año'],
    y=df_triangulo['Gasto Protección Ambiental (Millones MXN)'],
    name='Presupuesto Ambiental (INEGI)',
    marker_color='#2ca02c'
))

# 2. Multa Máxima de la Ley (Naranja)
fig.add_trace(go.Bar(
    x=df_triangulo['año'],
    y=df_triangulo['Multa Máxima LFRA (Millones MXN)'],
    name='Multa Máxima (LFRA)',
    marker_color='#ff7f0e'
))

# 3. Costo Conservador ITOPF (Rojo claro)
fig.add_trace(go.Bar(
    x=df_triangulo['año'],
    y=df_triangulo['Costo Limpieza ITOPF (Millones MXN)'],
    name='Daño Conservador (ITOPF)',
    marker_color='#ef553b'
))

# 4. Costo Integral EPA (Rojo oscuro - El Monstruo)
fig.add_trace(go.Bar(
    x=df_triangulo['año'],
    y=df_triangulo['Costo Restauración EPA (Millones MXN)'],
    name='Daño Integral (EPA)',
    marker_color='#8b0000'
))

fig.update_layout(
    title='<b>Costos En Economía:</b> Presupuesto Ambiental del país vs. Multa Aplicada vs. Daño Real en millones de pesos',
    xaxis_title='Año',
    yaxis_title='Millones de Pesos (MXN) - Escala Logarítmica', # Actualizamos el título
    yaxis_type='log', # <--- AQUÍ ESTÁ LA MAGIA MATEMÁTICA
    barmode='group',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="center",
        x=0.5
    )
)

fig.show()

Sección de Daño ambiental

Para esta parte nos basamos en: https://geoportal.semarnat.gob.mx/arcgisp/apps/webappviewer/index.html?id=6633959897c14220b90f1b92a6e11840


In [ ]:
import plotly.graph_objects as go

# Datos oficiales SEMARNAT (Corte a Dic 2025)
etiquetas = ['Sitios Remediados<br>(Daño reparado)', 'Sitios Contaminados<br>(Pasivos Ambientales Activos)']
valores = [1049, 1144]
colores = ['#2ca02c', '#8b0000'] # Verde para lo limpio, Rojo oscuro para el daño activo

fig = go.Figure(data=[go.Pie(
    labels=etiquetas,
    values=valores,
    hole=.5,
    marker=dict(colors=colores, line=dict(color='#000000', width=2)),
    textinfo='label+percent+value',
    textposition='outside',
    textfont_size=14
)])

fig.update_layout(
    title_text='<b>Daño Ambiental en México:</b><br>Inventario Nacional de Sitios Contaminados (SEMARNAT, 2025)',
    title_x=0.5,
    annotations=[dict(text='<b>Principal Causa:<br>Derrames de<br>Hidrocarburos</b>', x=0.5, y=0.5, font_size=14, showarrow=False)],
    showlegend=False,
    template='plotly_white'
)

fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# 1. Indicador Numérico
fig.add_trace(go.Indicator(
    mode = "number+delta",
    value = 1114,
    number = {'font': {'size': 80, 'color': '#8b0000'}},
    title = {"text": "<br><span style='font-size:1.2em;color:gray'>Sitios Contaminados Reconocidos<br>(Corte 2025)</span>"},
    domain = {'x': [0, 0.45], 'y': [0, 1]}
))

# 2. Texto de SEMARNAT en recuadro derecho
texto_medicina = (
    "<b>Documentación de SEMARNAT </b><br>"
    "<span style='font-size:11px; color:gray'>Reglamento y Portal Oficial de SEMARNAT</span><br><br>"
    "<i>'La información del Inventario se obtiene a partir de la gestión<br>"
    "de trámites... Para registrar un sitio contaminado, es necesario<br>"
    "tener la certeza de su contaminación, a través de resultados<br>"
    "analíticos emitidos por un laboratorio que así lo demuestren.'</i><br><br>"
    "<b> Sobre la métrica:</b><br>"
    "Esta métrica <b>no</b> representa el total del daño ambiental en el país.<br>"
    "Representa únicamente los derrames donde el infractor decidió<br>"
    "realizar el trámite administrativo y pagar un laboratorio."
)

# 3. La NOTA textual e íntegra sobre el -999999
nota_textual = (
    "<b>NOTA OFICIAL:</b> Los campos de área y volumen contaminado o remediado tienen valores<br>"
    "que no deben considerarse para alguna estadística ya que significa:<br>"
    "<span style='color:#8b0000; font-size:14px'><b>-999999 = No especifica información</b></span>"
)

fig.update_layout(
    template='plotly_white',
    height=420, # Ajuste de altura para que respiren los textos
    margin=dict(l=20, r=20, t=40, b=100), # Margen inferior más grande para la nota
    annotations=[
        # La nota del -999999 colocada exactamente debajo del indicador
        dict(
            x=0.22,
            y=-0.25, # Coordenada negativa para ponerla debajo del gráfico
            showarrow=False,
            text=nota_textual,
            font=dict(size=12, color="#555555"),
            align="center",
            xref="paper", yref="paper"
        ),
        # El recuadro del lado derecho
        dict(
            x=1.0,
            y=0.5,
            showarrow=False,
            text=texto_medicina,
            font=dict(size=13, color="#333333"),
            align="left",
            bgcolor="#f8f9fa",
            bordercolor="#555555",
            borderwidth=2,
            borderpad=15,
            xref="paper", yref="paper"
        )
    ]
)

fig.show()